# 믿:음 2.0 Base · Colab 시연용 OpenAI 호환 서버

이 노트북은 `K-intelligence/Midm-2.0-Base-Instruct`를 **4-bit로 Colab GPU에 로드**하고, 현재 가족센터 FastAPI가 호출할 수 있는 `/v1/chat/completions` API를 엽니다.

- 시연·합성 페르소나 데이터만 사용하세요. 실제 내담자 개인정보는 보내지 마세요.
- Colab 탭과 런타임이 살아 있는 동안만 동작하며, 재연결하면 터널 URL이 바뀝니다.
- 먼저 `런타임 > 런타임 유형 변경 > T4 GPU` 이상을 선택하세요.
- Colab 왼쪽의 **열쇠(Secrets)** 에 `NGROK_AUTHTOKEN`을 추가하고 노트북 액세스를 허용하세요. `HF_TOKEN`은 다운로드 제한이 생길 때만 선택적으로 추가합니다.
- ngrok 무료 계정과 authtoken은 https://dashboard.ngrok.com/get-started/your-authtoken 에서 준비합니다.


## 1. 패키지 설치
처음 한 번 수 분이 걸릴 수 있습니다. 설치 후 런타임을 재시작하라는 메시지가 나와도 우선 다음 셀을 실행해 보세요.


In [ ]:
%pip -q install -U "transformers>=4.49,<5" "accelerate>=1.2,<2" "bitsandbytes>=0.45,<1" "fastapi>=0.115,<1" "uvicorn[standard]>=0.34,<1" "ngrok>=1.4,<2" "pydantic>=2.10,<3"


## 2. GPU와 Secret 확인
`MIDM_API_KEY` Secret은 선택 사항입니다. 없으면 이 런타임 전용 키를 자동 생성해 마지막에 출력합니다.


In [ ]:
import os
import secrets
import torch
from google.colab import userdata

MODEL_ID = "K-intelligence/Midm-2.0-Base-Instruct"
SERVER_PORT = 8000
MAX_INPUT_TOKENS = 6144
MAX_OUTPUT_TOKENS = 1600

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. 런타임 > 런타임 유형 변경에서 T4 GPU 이상을 선택하세요.")

def optional_secret(name: str) -> str:
    try:
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

NGROK_AUTHTOKEN = optional_secret("NGROK_AUTHTOKEN")
HF_TOKEN = optional_secret("HF_TOKEN") or None
MIDM_API_KEY = optional_secret("MIDM_API_KEY") or secrets.token_urlsafe(32)
if not NGROK_AUTHTOKEN:
    raise RuntimeError("Colab Secrets에 NGROK_AUTHTOKEN을 추가하고 노트북 액세스를 켜세요.")

gpu_name = torch.cuda.get_device_name(0)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {gpu_name}")
print(f"4-bit 계산 dtype: {compute_dtype}")
print("Secret 확인 완료 (실제 값은 표시하지 않음)")


## 3. 믿:음 Base 4-bit 로드
최초 실행은 약 23GB 원본 가중치를 내려받고 양자화하며, 환경에 따라 5~20분 정도 걸릴 수 있습니다. 완료 메시지가 나올 때까지 기다리세요.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f"모델 로드 완료 · GPU allocated {allocated:.1f}GB / reserved {reserved:.1f}GB")


## 4. OpenAI 호환 API 정의 및 로컬 서버 시작
동시 생성은 GPU 메모리 급증을 피하기 위해 1건씩 처리합니다. 입력은 6,144토큰, 출력은 1,600토큰으로 제한합니다.


In [ ]:
import asyncio
import threading
import time
import uuid
from typing import Literal

import uvicorn
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Mi:dm 2.0 Base Colab Demo Server", version="0.1.0")
generation_lock = threading.Lock()

class ChatMessage(BaseModel):
    role: Literal["system", "user", "assistant"]
    content: str = Field(min_length=1, max_length=50000)

class ChatCompletionRequest(BaseModel):
    model: str = MODEL_ID
    messages: list[ChatMessage] = Field(min_length=1, max_length=50)
    max_tokens: int = Field(default=900, ge=1, le=MAX_OUTPUT_TOKENS)
    temperature: float = Field(default=0.35, ge=0.0, le=2.0)
    top_p: float = Field(default=0.9, gt=0.0, le=1.0)
    stream: bool = False

def require_api_key(authorization: str | None = Header(default=None)) -> None:
    expected = f"Bearer {MIDM_API_KEY}"
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Invalid API key")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID, "gpu": gpu_name}

@app.get("/v1/models", dependencies=[Depends(require_api_key)])
def models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model", "owned_by": "K-intelligence"}]}

def generate_sync(request: ChatCompletionRequest):
    messages = [item.model_dump() for item in request.messages]
    with generation_lock:
        batch = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        # Mi:dm tokenizer가 token_type_ids를 반환해도 모델에는 전달하지 않습니다.
        # 지원하는 텐서만 명시적으로 선택해 Transformers 버전 차이도 흡수합니다.
        batch = {key: value for key, value in batch.items() if key in {"input_ids", "attention_mask"}}
        input_tokens = int(batch["input_ids"].shape[-1])
        if input_tokens > MAX_INPUT_TOKENS:
            raise ValueError(f"입력이 {input_tokens}토큰입니다. {MAX_INPUT_TOKENS}토큰 이하로 줄이세요.")
        batch = {key: value.to(model.device) for key, value in batch.items()}
        generation_args = {
            "max_new_tokens": request.max_tokens,
            "do_sample": request.temperature > 0,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True,
        }
        if request.temperature > 0:
            generation_args.update(temperature=max(0.01, request.temperature), top_p=request.top_p)
        with torch.inference_mode():
            output = model.generate(**batch, **generation_args)
        generated = output[0, input_tokens:]
        text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        # greedy 생성이 첫 토큰에서 EOS로 끝나는 경우 한 번만 sampling으로 재시도합니다.
        if not text and not generation_args["do_sample"]:
            retry_args = {**generation_args, "do_sample": True, "temperature": 0.35, "top_p": 0.9}
            with torch.inference_mode():
                output = model.generate(**batch, **retry_args)
            generated = output[0, input_tokens:]
            text = tokenizer.decode(generated, skip_special_tokens=True).strip()
        if not text:
            token_preview = generated[:16].detach().cpu().tolist()
            raise ValueError(f"모델이 빈 응답을 생성했습니다. 생성 토큰: {token_preview}")
        return text, input_tokens, int(generated.shape[-1])

@app.post("/v1/chat/completions", dependencies=[Depends(require_api_key)])
async def chat_completions(request: ChatCompletionRequest):
    if request.stream:
        raise HTTPException(status_code=400, detail="이 시연 서버는 stream=false만 지원합니다.")
    try:
        text, prompt_tokens, completion_tokens = await asyncio.to_thread(generate_sync, request)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": MODEL_ID,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": text}, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": prompt_tokens + completion_tokens},
    }

if "uvicorn_server" in globals():
    uvicorn_server.should_exit = True
    if "server_thread" in globals() and server_thread.is_alive():
        server_thread.join(timeout=10)
    if "server_thread" in globals() and server_thread.is_alive():
        raise RuntimeError("이전 API 서버가 아직 종료되지 않았습니다. 5초 후 이 셀만 다시 실행하세요.")
uvicorn_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=SERVER_PORT, log_level="warning"))
server_thread = threading.Thread(target=uvicorn_server.run, daemon=True)
server_thread.start()
deadline = time.time() + 15
while not uvicorn_server.started and server_thread.is_alive() and time.time() < deadline:
    time.sleep(0.1)
if not uvicorn_server.started:
    raise RuntimeError(f"로컬 API가 포트 {SERVER_PORT}에서 시작되지 않았습니다. 서버 셀 출력을 확인하세요.")
print(f"로컬 API 시작: http://127.0.0.1:{SERVER_PORT} · 최신 서버 코드 적용됨")


## 5. 로컬 API 1차 테스트
첫 생성은 CUDA 준비 때문에 이후 요청보다 느릴 수 있습니다. 응답 내용이 출력되면 모델과 API가 정상입니다.


In [ ]:
import requests

headers = {"Authorization": f"Bearer {MIDM_API_KEY}", "Content-Type": "application/json"}
test_payload = {
    "model": MODEL_ID,
    "messages": [
        {"role": "system", "content": "너는 한국어로 간결하게 답하는 테스트 도우미다."},
        {"role": "user", "content": "연결 확인이라고 짧게 답해줘."},
    ],
    "max_tokens": 48,
    "temperature": 0.0,
    "stream": False,
}
response = requests.post(f"http://127.0.0.1:{SERVER_PORT}/v1/chat/completions", headers=headers, json=test_payload, timeout=180)
if not response.ok:
    try:
        error_body = response.json()
    except ValueError:
        error_body = response.text
    raise RuntimeError(
        f"믿:음 로컬 API HTTP {response.status_code}: {error_body}\n"
        "위의 FastAPI 서버 셀을 다시 실행한 뒤 이 테스트 셀을 재실행하세요."
    )
print(response.json()["choices"][0]["message"]["content"])


## 6. ngrok 임시 HTTPS 주소 열기
셀 출력의 `.env` 블록을 로컬 프로젝트 루트의 `.env`에 그대로 반영합니다. API 키가 있으므로 URL만 알아서는 생성 API를 호출할 수 없습니다.


In [ ]:
import inspect
import ngrok

if "ngrok_listener" in globals():
    try:
        existing_listener = await ngrok_listener if inspect.isawaitable(ngrok_listener) else ngrok_listener
        close_result = existing_listener.close()
        if inspect.isawaitable(close_result):
            await close_result
    except Exception:
        pass
forward_result = ngrok.forward(SERVER_PORT, authtoken=NGROK_AUTHTOKEN)
ngrok_listener = await forward_result if inspect.isawaitable(forward_result) else forward_result
PUBLIC_URL = ngrok_listener.url().rstrip("/")
OPENAI_BASE_URL = f"{PUBLIC_URL}/v1"

print("\n===== 로컬 프로젝트 .env에 넣을 값 =====")
print("AI_PROVIDER=internal_openai")
print(f"INTERNAL_LLM_BASE_URL={OPENAI_BASE_URL}")
print(f"INTERNAL_LLM_MODEL={MODEL_ID}")
print(f"INTERNAL_LLM_API_KEY={MIDM_API_KEY}")
print("LLM_REQUEST_TIMEOUT=240")
print("LLM_HEALTH_TIMEOUT=12")
print("=========================================\n")
print("중요: Colab 런타임이 재연결되면 이 셀을 다시 실행하고 새 URL로 .env를 갱신하세요.")


## 7. 외부 주소 최종 테스트
여기까지 성공하면 Windows의 현재 FastAPI도 같은 주소를 호출할 수 있습니다.


In [ ]:
public_headers = {**headers, "ngrok-skip-browser-warning": "1"}
models_response = requests.get(f"{OPENAI_BASE_URL}/models", headers=public_headers, timeout=30)
models_response.raise_for_status()
print("외부 연결 정상:", models_response.json()["data"][0]["id"])


## 8. Windows 앱 연결
1. 위 출력값을 프로젝트 루트 `.env`에 붙여 넣습니다. 기존 키가 있으면 해당 줄을 교체합니다.
2. 실행 중인 **백엔드 PowerShell에서 `Ctrl+C`** 후 프로젝트 루트에서 다시 실행합니다.
```powershell
.\.venv\Scripts\python.exe -m pip install -r backend\requirements.txt
.\.venv\Scripts\python.exe -m uvicorn backend.app.main:app --host 127.0.0.1 --port 8100
```
3. 프론트엔드는 재시작하지 않아도 됩니다. `http://127.0.0.1:3000/training` 또는 상담 코파일럿 화면을 새로고침합니다.
4. 상단에 **믿:음 연결 정상**이 보이면 완료입니다. 오프라인이면 이 노트북의 6·7번 셀과 `.env` URL을 확인하세요.

Colab 탭을 닫거나 런타임이 종료되면 기존 화면은 유지되지만 새 AI 응답 생성은 실패합니다. 시연 시작 전에 7번 셀을 한 번 실행해 상태를 확인하세요.
